<a href="https://colab.research.google.com/github/chandinignits-del/ML-Lab-GNITS-2026-27/blob/main/Decision_Tree_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import math

# ============================================================
# 1. ENTROPY
# ============================================================
def entropy(data, target):
    if len(data) == 0:
        return 0

    counts = data[target].value_counts()
    result = 0

    for count in counts:
        p = count / len(data)
        result -= p * math.log2(p)

    return result


# ============================================================
# 2. INFORMATION GAIN
# ============================================================
def information_gain(data, attribute, target):
    total_entropy = entropy(data, target)
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]

        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset, target)

    return total_entropy - weighted_entropy


# ============================================================
# 3. ID3 ALGORITHM
# ============================================================
def id3(data, original_data, features, target):

    # Case 1: all examples belong to one class
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    # Case 2: dataset is empty
    if len(data) == 0:
        return original_data[target].mode()[0]

    # Case 3: no attributes are left
    if len(features) == 0:
        return data[target].mode()[0]

    # Calculate information gain for every attribute
    gains = []

    for feature in features:
        gains.append(information_gain(data, feature, target))

    # Select attribute with maximum information gain
    best_feature = features[gains.index(max(gains))]

    tree = {best_feature: {}}

    remaining_features = [
        f for f in features if f != best_feature
    ]

    # Create branches
    for value in data[best_feature].unique():

        subset = data[data[best_feature] == value]

        subtree = id3(
            subset,
            original_data,
            remaining_features,
            target
        )

        tree[best_feature][value] = subtree

    return tree


# ============================================================
# 4. PREDICTION USING THE TREE
# ============================================================
def predict(tree, row, default_class="No"):

    # Leaf node
    if not isinstance(tree, dict):
        return tree

    attribute = list(tree.keys())[0]
    value = row[attribute]

    # If an unseen value occurs, use default class
    if value not in tree[attribute]:
        return default_class

    return predict(
        tree[attribute][value],
        row,
        default_class
    )


# ============================================================
# 5. CALCULATE ACCURACY
# ============================================================
def calculate_accuracy(tree, data, target):

    if len(data) == 0:
        return 0

    default_class = data[target].mode()[0]
    correct = 0

    for _, row in data.iterrows():

        prediction = predict(tree,row,default_class)

        if prediction == row[target]:
            correct += 1

    return correct / len(data)


# ============================================================
# 6. REDUCED ERROR PRUNING
# ============================================================
def prune_tree(tree, validation_data, target):

    # Already a leaf
    if not isinstance(tree, dict):
        return tree

    # No validation examples reach this node
    if len(validation_data) == 0:
        return tree

    attribute = list(tree.keys())[0]

    # --------------------------------------------------------
    # First prune child nodes
    # --------------------------------------------------------
    for value in list(tree[attribute].keys()):

        subset = validation_data[validation_data[attribute] == value]

        tree[attribute][value] = prune_tree(
            tree[attribute][value],
            subset,
            target
        )

    # --------------------------------------------------------
    # Accuracy BEFORE pruning
    # --------------------------------------------------------
    accuracy_before = calculate_accuracy(
        tree,
        validation_data,
        target
    )

    # Majority class becomes the replacement leaf
    majority_class = validation_data[target].mode()[0]

    # --------------------------------------------------------
    # Accuracy AFTER replacing subtree with a leaf
    # --------------------------------------------------------
    correct = 0

    for _, row in validation_data.iterrows():

        if row[target] == majority_class:
            correct += 1

    accuracy_after = correct / len(validation_data)

    print("\nChecking node:", attribute)
    print("Majority class:", majority_class)
    print(
        "Accuracy before pruning:",
        round(accuracy_before * 100, 2),
        "%"
    )
    print(
        "Accuracy after pruning:",
        round(accuracy_after * 100, 2),
        "%"
    )

    # --------------------------------------------------------
    # Prune if accuracy improves or remains the same
    # --------------------------------------------------------
    if accuracy_after >= accuracy_before:

        print(">>> Node", attribute, "PRUNED")

        return majority_class

    else:

        print(">>> Node", attribute, "KEPT")

        return tree


# ============================================================
# 7. PRINT DECISION TREE
# ============================================================
def print_tree(tree, indent=""):

    if not isinstance(tree, dict):

        print(indent + "-> " + str(tree))

        return

    attribute = list(tree.keys())[0]

    for value in tree[attribute]:

        print(
            indent
            + attribute
            + " = "
            + str(value)
        )

        print_tree(
            tree[attribute][value],
            indent + "    "
        )


# ============================================================
# 8. MAIN PROGRAM
# ============================================================
df = pd.read_csv("tennisdata.csv")

print("==============================================")
print("             PLAY TENNIS DATASET")
print("==============================================\n")

print(df)

target = "PlayTennis"

features = [
    column for column in df.columns
    if column != target
]

# ------------------------------------------------------------
# Split data into training and validation sets
# ------------------------------------------------------------
# For this small classroom dataset, the first 10 records
# are used for training and the remaining 4 for validation.
training_data = df.iloc[:10].copy()
validation_data = df.iloc[10:].copy()

print("\n==============================================")
print("TRAINING DATA")
print("==============================================")
print(training_data)

print("\n==============================================")
print("VALIDATION DATA")
print("==============================================")
print(validation_data)

# ------------------------------------------------------------
# Build ID3 tree
# ------------------------------------------------------------
tree = id3(
    training_data,
    training_data,
    features,
    target
)

print("\n==============================================")
print("ID3 DECISION TREE BEFORE PRUNING")
print("==============================================")

print_tree(tree)

# ------------------------------------------------------------
# Accuracy before pruning
# ------------------------------------------------------------
training_accuracy_before = calculate_accuracy(
    tree,
    training_data,
    target
)

validation_accuracy_before = calculate_accuracy(
    tree,
    validation_data,
    target
)

print("\nTraining accuracy BEFORE pruning:",
      round(training_accuracy_before * 100, 2), "%")

print("Validation accuracy BEFORE pruning:",
      round(validation_accuracy_before * 100, 2), "%")

# ------------------------------------------------------------
# Reduced Error Pruning
# ------------------------------------------------------------
print("\n==============================================")
print("REDUCED ERROR PRUNING")
print("==============================================")

pruned_tree = prune_tree(
    tree,
    validation_data,
    target
)

# ------------------------------------------------------------
# Accuracy after pruning
# ------------------------------------------------------------
training_accuracy_after = calculate_accuracy(
    pruned_tree,
    training_data,
    target
)

validation_accuracy_after = calculate_accuracy(
    pruned_tree,
    validation_data,
    target
)

print("\n==============================================")
print("DECISION TREE AFTER PRUNING")
print("==============================================")

print_tree(pruned_tree)

print("\nTraining accuracy AFTER pruning:",
      round(training_accuracy_after * 100, 2), "%")

print("Validation accuracy AFTER pruning:",
      round(validation_accuracy_after * 100, 2), "%")

             PLAY TENNIS DATASET

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No

TRAINING DATA
    Outlook Temperature Humidity    Wind PlayTennis
0     Sunny         Hot     High    Weak         No
1     Sunny         Hot     High  Strong         No
